In [1]:
import pandas as pd
import numpy as np
import os

def preprocess_dataset(file_path):
    print(f"Loading dataset from: {file_path}")
    # LOADING AS EXCEL
    df = pd.read_excel(file_path)
    
    # Filling actual NaNs with 'NULL' string to standardize the string matching for rules
    df = df.fillna('NULL')

    print(f"Original shape: {df.shape}")

    # ENGAGEMENT CATEGORY REPLACEMENTS
    engagement_map = {
        'Child Not in School': 'Child',
        'Infant': 'Child',
        'Child out of School': 'Child',
        'Landowner': 'Farmer',
        'Landowner Farmer': 'Farmer',
        'Industrial Labour': 'Private Sector Employee',
        'Private contractual worker': 'Private Sector Employee',
        'State Govt./PSU contractual Employee': 'Private Sector Employee',
        'Senior Citizens Old Age Pensioner': 'Senior Citizen',
        'Super Senior Citizen Social Security Pensioner': 'Senior Citizen',
        'Agricultural labour': 'Labour',
        'Construction worker': 'Labour',
        'Other labour': 'Labour',
        'State Government/ PSU Employee': 'Government Employee',
        'Central Government Employee': 'Government Employee',
        'Central PSU Employee': 'Government Employee',
        'Employee of Other State Government': 'Government Employee',
        '': 'Others',
        'NULL': 'Others'
    }
    
    # Specific logic (setting EmployeeTypee to 'Regular')
    gov_emp_list = ['State Government/ PSU Employee', 'Central Government Employee', 
                    'Central PSU Employee', 'Employee of Other State Government']
    df.loc[df['engagement'].isin(gov_emp_list), 'EmployeeTypee'] = 'Regular'
    
    # Applying mapping
    df['engagement'] = df['engagement'].replace(engagement_map)

    # CONDITIONAL ENGAGEMENT OVERRIDES
    df.loc[df['EmployeeTypee'] == 'Regular', 'engagement'] = 'Government Employee'
    df.loc[df['EmployeeTypee'] == 'Contractual', 'engagement'] = 'Government Contractual Employee'
    df.loc[(df['engagement'] == 'Government Employee') & (df['isGovPensioner'].astype(str) == '1'), 'engagement'] = 'Pensioner/Retired'
    df.loc[(df['engagement'] == 'Government Employee') & (df['isGovPensioner'].astype(str) == '1'), 'EmployeeTypee'] = 'NULL'
    df.loc[(df['engagement'] == 'Government Employee') & (df['isLabour'].astype(str) == '1'), 'isLabour'] = '0'
    df.loc[(df['engagement'] == 'Government Employee') & (df['isLabour'].astype(str) == '1'), 'labour_wages'] = 'NULL'
    df.loc[df['isLabour'].astype(str) == '1', 'engagement'] = 'Private Sector Employee'
    
    # ONE HOT ENCODING - ENGAGEMENT
    engagement_dummies = pd.get_dummies(df['engagement'], prefix='is').astype(int)
    df = pd.concat([df, engagement_dummies], axis=1)

    # MEMBER VERIFIED RANGE
    # df['memberVerifiedRange'] = df['memberVerifiedRange'].replace('NULL', '0')
    # verified_dummies = pd.get_dummies(df['memberVerifiedRange'], prefix='in').astype(int)
    # df = pd.concat([df, verified_dummies], axis=1)
    # Uncomment the above lines to get the separate columns for 'meberVerfiedRange'
    
    # DROPPING COLUMNS
    cols_to_drop = ['isGovEmp', 'EmpAnuualIncome', 'VehicleRegistrationDate', 'isEducationDataAvailabe', 'class']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # RENAMING COLUMNS
    rename_dict = {
        'isIncomeTaxPay': 'isIncomeTaxPayee',
        'isFourVehicle': 'isFourWheeler',
        'isElectrictyMapping': 'isElectricityMapping',
        'SchoolOwner': 'SchoolType',
        'EmployeeTypee': 'EmployeeType'
    }
    df = df.rename(columns=rename_dict)

    # ONE HOT ENCODING - EMPLOYEE TYPE
    emp_type_dummies = pd.get_dummies(df['EmployeeType'], prefix='is').astype(int)
    if 'is_NULL' in emp_type_dummies.columns:
        emp_type_dummies = emp_type_dummies.drop(columns=['is_NULL'])
    df = pd.concat([df, emp_type_dummies], axis=1)

    # INCOME TAX THRESHOLD
    temp_tax = pd.to_numeric(df['incometaxthreshold'], errors='coerce')
    condition_19 = (df['incometaxthreshold'] == 'NULL') | (temp_tax < 180000)
    df.loc[condition_19, 'isIncomeTaxPayee'] = 0
    df.loc[condition_19, 'incometaxthreshold'] = 'NULL'
    
    tax_dummies = pd.get_dummies(df['incometaxthreshold'], prefix='incometaxthreshold').astype(int)
    if 'incometaxthreshold_NULL' in tax_dummies.columns:
        tax_dummies = tax_dummies.drop(columns=['incometaxthreshold_NULL'])
    df = pd.concat([df, tax_dummies], axis=1)

    # CONDITIONAL VALUE IMPUTATIONS
    df.loc[(df['isGovPensioner'].astype(str) == '1') & (df['PenAmt'] == 'NULL'), 'PenAmt'] = 2500
    df.loc[(df['isGovPensioner'].astype(str) == '0') & (df['PenAmt'] == 'NULL'), 'PenAmt'] = 0
    
    df.loc[(df['isLabour'].astype(str) == '1') & (df['labour_wages'].astype(str) == '0'), 'labour_wages'] = 15000
    df.loc[(df['isLabour'].astype(str) == '0') & (df['labour_wages'] == 'NULL'), 'labour_wages'] = 0

    df.loc[(df['isElectricityMapping'].astype(str) == '1') & (df['electric_annualbillamount'].isin(['0', 'NULL'])), 'electric_annualbillamount'] = 15000
    df.loc[(df['isElectricityMapping'].astype(str) == '0') & (df['electric_annualbillamount'] == 'NULL'), 'electric_annualbillamount'] = 0

    df.loc[(df['isRuralProperty'].astype(str) == '1') & (df['PanchayatPropertyArea'].isin(['0.00', '0'])), 'PanchayatPropertyArea'] = 30
    df.loc[(df['isRuralProperty'].astype(str) == '0') & (df['PanchayatPropertyArea'] == 'NULL'), 'PanchayatPropertyArea'] = 0

    two_three_wheelers = ['M-Cycle/Scooter', 'Three Wheeler (Passenger)', 
                          'e-Rickshaw(P)', 'e-Rickshaw with Cart (G)', 'NULL', 'Moped', 
                          'Three Wheeler (Goods)', 'M-Cycle/Scooter-With Side Car', 'Harvester']
    df.loc[(df['VehicleClass'].isin(two_three_wheelers)) & (df['isFourWheeler'].astype(str) == '1'), 'isFourWheeler'] = 0

    df.loc[(df['isFarmer'].astype(str) == '1') & (df['Farmer_EarningAmount'] == 'NULL'), 'Farmer_EarningAmount'] = 30000

    # SCHOOL TYPE OHE
    df['age_numeric'] = pd.to_numeric(df['age'], errors='coerce').fillna(0)
    df.loc[(df['age_numeric'] > 25) & (df['SchoolType'] != 'NULL'), 'SchoolType'] = 'NULL'
    
    school_dummies = pd.get_dummies(df['SchoolType'], prefix='in').astype(int)
    if 'in_NULL' in school_dummies.columns:
        school_dummies = school_dummies.drop(columns=['in_NULL'])
    df = pd.concat([df, school_dummies], axis=1)

    # DROPPING REDUNDANT COLUMNS BEFORE GROUPING
    cols_to_drop_pre_group = ['hasmemberid', 'qualification', 'age', 'gender', 'age_numeric', 
                              'engagement', 'memberVerifiedRange', 'EmployeeType', 
                              'incometaxthreshold', 'SchoolType', 'VehicleClass']
    df = df.drop(columns=[col for col in cols_to_drop_pre_group if col in df.columns])

    # PRE-GROUPING CLEANUP (Converting amounts to numeric)
    numeric_cols = ['isIncomeTaxPayee', 'isGovPensioner', 'PenAmt', 'isLabour', 'labour_wages', 
                    'isElectricityMapping', 'electric_annualbillamount', 'isRuralProperty', 
                    'PanchayatPropertyArea', 'isUrbanProperty', 'isFourWheeler', 'isFarmer', 
                    'FarmerArea', 'Farmer_EarningAmount']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('NULL', '0'), errors='coerce').fillna(0)

    # GROUPING BY FAMILY ID
    agg_dict = {}
    metadata_cols = ['district', 'blocktown', 'wardvillage', 'r_u', 'familyRange']
    for col in metadata_cols:
        if col in df.columns:
            agg_dict[col] = 'first'
            
    ohe_columns = list(engagement_dummies.columns) + list(emp_type_dummies.columns) + list(tax_dummies.columns) + list(school_dummies.columns) \
    #              + list(verified_dummies.columns)
    # Uncomment the above line to get the separate columns for 'memberVerifiedRange'
    sum_columns = ohe_columns + numeric_cols
    
    for col in sum_columns:
        if col in df.columns:
            agg_dict[col] = 'sum'

    print("Grouping by Family ID and Property ID...")
    # Adding Property_ID here to group so it acts exactly like hasfamilyid
    df_final = df.groupby(['hasfamilyid', 'Property_ID'], as_index=False).agg(agg_dict)
    print(f"Final shape after grouping: {df_final.shape}")
    
    # EXPORTING AS EXCEL TO THE SAME DIRECTORY
    # Generating the output filename by adding 'processed_' to the original filename
    base_name = os.path.basename(file_path)
    dir_name = os.path.dirname(file_path)
    output_filename = os.path.join(dir_name, f"processed_{base_name}")
    
    print(f"Saving to Excel: {output_filename}")
    # SAVING AS EXCEL
    df_final.to_excel(output_filename, index=False)
    print("File saved successfully!")
    
    return df_final

final_df = preprocess_dataset('district_data_with_property_ids.xlsx')

Loading dataset from: district_data_with_property_ids.xlsx
Original shape: (9134, 36)


C:\Users\Harsh Datt\AppData\Local\Temp\ipykernel_18400\1682992559.py:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[(df['engagement'] == 'Government Employee') & (df['isLabour'].astype(str) == '1'), 'isLabour'] = '0'


Grouping by Family ID and Property ID...
Final shape after grouping: (2301, 45)
Saving to Excel: processed_district_data_with_property_ids.xlsx
File saved successfully!
